__Exploratory Data Analysis (EDA)__

- Inspect distributions, missing values, outliers.
- Plot time series of IV, skew, curvature.
- Compare SPY vs QQQ.
- Correlation checks.
- Document findings.

In [1]:
# import parquet_extractor  
# import importlib

# importlib.reload(parquet_extractor) 

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from parquet_extractor import load_ticker_year_data, get_ticker_metadata

__Load the filtered parquet__

In [3]:
# metadata = get_ticker_metadata('SPY', Path("."))
# metadata

In [4]:
def load_merged_ticker_data(tickers, start_year, end_year, data_dir=Path(".")):
    """Load and merge data for multiple tickers across a range of years."""
    all_data = []

    if isinstance(tickers, str):
        tickers = [tickers]

    for ticker in tickers:
        for year in range(start_year, end_year + 1):
            try:
                df = load_ticker_year_data(ticker, year, data_dir)
                df = df.with_columns([
                    pl.lit(ticker).alias("ticker"),
                ])
                all_data.append(df)
            except FileNotFoundError:
                print(f"Data not found for {ticker} in {year}, skipping.")
            except Exception as e:
                print(f"Error loading {ticker} in {year}: {e}")

    if all_data:
        return pl.concat(all_data)
    else:
        return pl.DataFrame()

In [5]:
spy_pl = load_merged_ticker_data('SPY', 2005, 2023, Path("."))
spy_df = spy_pl.to_pandas()
spy_df.sample(7)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
16812619,2020-12-29,109820.0,SPY 210319P380000,P,2021-03-19,380000.0,17.96,18.03,187.0,40630.0,...,0.00000,0.00000,2.45999,2.45999,4.22000,-0.78000,1.86999,1.60998,16.13000,SPY
6751,2005-09-02,109820.0,CYY.LA,C,2006-12-16,105000.0,21.80,22.00,10.0,366.0,...,1.22000,1.22000,0.68000,-0.79000,-1.06000,-1.38000,-1.17000,1.78000,6.55000,SPY
17047606,2021-07-06,109820.0,SPY 211217P115000,P,2021-12-17,115000.0,0.10,0.11,861.0,5441.0,...,-0.79001,-0.79001,2.50000,4.86999,9.82000,8.44998,22.07000,41.44998,57.22998,SPY
19664595,2021-12-31,109820.0,SPY 220121C454000,C,2022-01-21,454000.0,22.65,22.76,49.0,4237.0,...,-1.20001,-1.20001,-1.20001,-2.52002,-1.91001,-2.30002,11.89999,4.35998,16.16998,SPY
16985007,2020-12-24,109820.0,SPY 210521C450000,C,2021-05-21,450000.0,0.27,0.30,20.0,710.0,...,0.00000,1.42999,1.76001,1.14001,-3.23999,4.34000,-1.17001,11.54001,38.79999,SPY
1400236,2009-07-20,109820.0,OBM.XE,P,2011-12-17,25000.0,0.30,0.35,31.0,24683.0,...,1.00100,1.00100,1.87100,5.02900,6.96100,3.18100,3.09100,4.62100,7.74100,SPY
17325214,2021-05-17,109820.0,SPY 211217P290000,P,2021-12-17,290000.0,3.70,3.80,1.0,4033.0,...,-1.06000,-1.06000,5.23999,10.10999,1.31000,-6.60001,-1.78000,3.34998,14.91000,SPY


In [6]:
# Save to Parquet
spy_pl.write_parquet("./parquet/spy_historical_2005_2023_v1.parquet")

In [8]:
df = pl.read_parquet("./parquet/spy_historical_2005_2023_v1.parquet")
print(df.shape)
# print(df.head()

(24914760, 37)


In [10]:
df.sample(20)

date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,delta,vega,theta,forward_price,expiry_indicator,prc,vol,iv_30d,year,vol_delta_product,moneyness,volume_ma5,iv_rolling_std,price_diff_1d,price_diff_2d,price_diff_3d,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
date,f64,str,str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2008-05-14,109820.0,"""JBG.FT""","""C""",2008-06-30,150000.0,0.27,0.3,52.0,2005.0,0.13213,0.094724,8.389605,-4.835775,null,"""m""",140.77,1.819107e8,0.289517,2008,0.012516,0.938467,23.8,0.008855,0.29,0.29,0.29,0.29,0.29,0.29,0.30999,1.25,-0.06,1.14,3.91999,5.91999,"""SPY"""
2010-12-17,109820.0,"""SPY 101231P119000""","""P""",2010-12-31,119000.0,0.18,0.19,2526.0,25713.0,0.171288,-0.09368,4.074697,-9.065131,null,"""m""",124.3,1.410752e8,0.190809,2010,-0.016046,1.044538,2526.0,0.002144,0.0,-0.52,-0.52,-0.52,-0.52,0.202,0.202,-0.26,-0.18,1.47,3.29,4.01,"""SPY"""
2021-11-26,109820.0,"""SPY 211129C446000""","""C""",2021-11-29,446000.0,14.16,14.3,20.0,45.0,0.333588,0.832207,10.43713,-212.173,null,"""w""",458.97,1.12669636e8,0.23604,2021,0.277614,1.029081,21.2,0.045266,0.0,0.0,0.0,-10.47,-10.47,-10.47,-9.22,-8.60001,-10.76001,-10.31,-4.65,0.64999,"""SPY"""
2021-06-09,109820.0,"""SPY 210630C383000""","""C""",2021-06-30,383000.0,38.98,39.32,4.0,464.0,0.277782,0.947902,10.96698,-32.40065,null,"""m""",421.64999,4.8436342e7,0.218075,2021,0.26331,1.100914,4.0,0.131968,0.0,0.0,-0.63001,-0.63001,-0.63001,-0.63001,-0.95002,2.88,1.32,2.57998,9.70999,6.02999,"""SPY"""
2011-03-07,109820.0,"""SPY 110319C120000""","""C""",2011-03-19,120000.0,11.38,11.6,166.0,44786.0,0.264147,0.975999,1.285284,-5.701422,null,null,131.42999,2.168889e8,0.238902,2011,0.257807,1.09525,166.0,0.018105,0.0,0.0,0.0,0.0,0.0,-1.04001,-1.04001,-2.04001,0.5,0.5,-2.42002,-1.14002,"""SPY"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015-02-05,109820.0,"""SPY 150417C207000""","""C""",2015-04-17,207000.0,4.59,4.64,1425.0,4239.0,0.148194,0.476631,35.6378,-14.96504,null,null,206.12,9.7953181e7,0.136734,2015,0.070634,0.995749,1192.8,0.005642,0.0,0.0,2.06,2.06,2.06,2.06,1.28,4.2,4.1298,3.37999,3.04,3.47001,"""SPY"""
2020-07-02,109820.0,"""SPY 220121P270000""","""P""",2022-01-21,270000.0,22.55,23.94,1.0,875.0,0.269104,-0.290205,131.3844,-11.09816,null,null,312.23001,6.9344217e7,0.361836,2020,-0.078095,1.156407,549.8,0.063178,0.0,0.0,3.87002,3.87002,3.87002,7.77002,7.77002,0.18002,11.62002,-7.10999,9.26001,30.63,"""SPY"""
2020-04-14,109820.0,"""SPY 200515C390000""","""C""",2020-05-15,390000.0,0.01,0.02,17.0,1995.0,0.367943,0.001758,0.466396,-1.007272,null,null,283.79001,1.3414335e8,0.216212,2020,0.000647,0.727667,14.0,0.097129,0.0,0.0,0.0,0.0,5.59,5.59,31.96001,37.64002,40.64002,14.47,-16.44998,-48.41,"""SPY"""
